# Google Scholar Collection Pipeline

*Author: Regina Chua*

> This notebook supplements the PubMed pipeline by pulling results from Google
> Scholar. Unlike PubMed there is **no official Google Scholar API**. The `scholarly` package
> scrapes results directly, so Google will rate-limit or block the session after ~50–100 results
> without a proxy..
>
> **Reproducibility caveat:** Google personalises and reorders results between sessions, so two
> runs of this notebook may not return identical records.

This notebook follows the same section structure as `pubmed.ipynb`. All search criteria are
imported from [`search_strategy.py`](search_strategy.py) so the two pipelines stay in sync.

## 1. Environment Setup

> Same imports as the PubMed notebook plus `scholarly`, `time`, and
> `random`. The random sleep between requests is the main tool for avoiding a block.

In [1]:
import json
import random
import time
from datetime import datetime
from pathlib import Path

import pandas as pd

try:
    from scholarly import scholarly, MaxTriesExceededException
    print("scholarly loaded OK")
except ImportError:
    raise ImportError(
        "scholarly is not installed. Run:  pip install scholarly"
    )

from search_strategy import (
    INCLUSION_CRITERIA,
    ALTERNATE_TERMS,
    EXCLUSION_TERMS,
    DATE_FILTER,
)

pd.set_option("display.max_colwidth", 120)
print("Environment ready.")

scholarly loaded OK
Environment ready.


## 2. Build Query

> Google Scholar does not support boolean field syntax like PubMed's
> `[TIAB]` or `NOT` operators at the API level. Instead I build a flat OR-joined string from
> the primary inclusion terms and pass the date range through `scholarly`'s `year_low` /
> `year_high` parameters. Exclusion terms are **applied post-hoc** on the collected titles and
> abstracts (see Section 5), not server-side. I use the core `INCLUSION_CRITERIA` only (not
> `ALTERNATE_TERMS`).

In [2]:
def build_scholar_query(inclusion):
    """Build a flat OR-joined query string for Google Scholar.

    Scholar does not support NOT operators or field specifiers, so we join
    the primary disease, spatial, and exposure terms with OR, wrapped in
    quotes. Exclusion filtering happens post-hoc in Section 5.
    """
    all_terms = (
        inclusion["disease"]
        + inclusion["spatial"]
        + inclusion["exposure"]
    )
    # Drop wildcard characters — Scholar treats * literally
    cleaned = [t.replace("*", "") for t in all_terms]
    return " OR ".join(f'"{t}"' for t in cleaned)


query = build_scholar_query(INCLUSION_CRITERIA)
year_low  = int(DATE_FILTER["start_date"][:4])
year_high = int(DATE_FILTER["end_date"][:4])

print(f"Year range: {year_low} – {year_high}")
print(f"\nQuery ({query.count('OR') + 1} terms):\n{query}")

Year range: 2020 – 2025

Query (17 terms):
"parkinson disease" OR "neurodegenerative disease" OR "geospatial" OR "spatial dependence" OR "spatiotemporal" OR "geographic" OR "environment" OR "atmospheric" OR "spatial analysis" OR "pollution" OR "chemical" OR "pesticide" OR "air pollution" OR "microplastic pollution" OR "traffic pollution" OR "water pollution" OR "trichloroethylene"


## 3. Rate-Limiting & Proxy Note

> Google will serve
> a CAPTCHA or block the IP after a number of rapid requests. The mitigations below are in
> increasing order of effort:
>
> 1. **Random sleep (default, already in place)** — 2–6 s between records.
> 2. **Tor proxy** — free but requires Tor running locally:
>    ```python
>    from scholarly import ProxyGenerator
>    pg = ProxyGenerator()
>    pg.Tor_Internal(tor_cmd="tor")
>    scholarly.use_proxy(pg)
>    ```
> 3. **ScraperAPI / SerpApi** — paid, reliable, no setup:
>    ```python
>    pg = ProxyGenerator()
>    pg.ScraperAPI("YOUR_KEY")
>    scholarly.use_proxy(pg)
>    ```
>
> If the run is blocked, the notebook catches `MaxTriesExceededException` and saves whatever
> was collected up to that point.

## 4. Collect Articles

> I iterate the `scholarly` generator manually with `next()` so I can
> inject the sleep and handle blocking. `scholarly.fill()` fetches the full abstract
> for each result.

In [ ]:
MAX_RESULTS = 150  # Conservative limit — increase with a proxy in place
SLEEP_MIN   = 2.0  # seconds between requests
SLEEP_MAX   = 6.0

def flatten_pub(pub):
    """Flatten a scholarly publication dict into a row matching the PubMed schema."""
    bib  = pub.get("bib", {})
    year = bib.get("pub_year", None)
    return {
        "title":            bib.get("title", None),
        "abstract":         bib.get("abstract", None),
        "publication_date": f"{year}-01-01" if year else None,
        "authors":          bib.get("author", None),
        "journal":          bib.get("venue", None),
        "doi":              pub.get("pub_url", None),   # Scholar rarely has a DOI field
        "url":              pub.get("pub_url", None),
        "num_citations":    pub.get("num_citations", None),
        # Fields not available from Scholar
        "pubmed_id":        None,
        "keywords":         None,
        "source":           "google_scholar",
    }


records = []
search_gen = scholarly.search_pubs(query, year_low=year_low, year_high=year_high)
run_ts = datetime.now().isoformat(timespec="seconds")

print(f"Starting collection at {run_ts}  (max {MAX_RESULTS} results) ...")
print("Google may block the session — the notebook will save partial results if that happens.\n")

try:
    for i in range(MAX_RESULTS):
        try:
            pub = next(search_gen)
        except StopIteration:
            print(f"Generator exhausted after {i} results.")
            break

        # Fetch full record (abstract etc.) — costs one extra request
        try:
            pub = scholarly.fill(pub)
        except Exception:
            pass  # Use whatever partial data we have

        records.append(flatten_pub(pub))

        delay = random.uniform(SLEEP_MIN, SLEEP_MAX)
        if (i + 1) % 10 == 0:
            print(f"  {i + 1} records collected … (sleeping {delay:.1f}s)")
        time.sleep(delay)

except MaxTriesExceededException:
    print(
        f"\nGoogle blocked the session after {len(records)} records.\n"
        "Partial results saved. To collect more:\n"
        "  - Wait 30–60 min and re-run\n"
        "  - Or configure a proxy (see Section 3)"
    )

df_raw = pd.DataFrame(records)
print(f"\nCollected {len(df_raw)} records total.")
preview_cols = [c for c in ["title", "abstract", "publication_date", "journal"] if c in df_raw.columns]
display(df_raw[preview_cols].head())

## 5. Post-hoc Exclusion Filter

> Because Scholar's query language has no NOT operator, I apply the
> exclusion terms here against the title and abstract text. This is less precise than PubMed's
> server-side NOT but is the best available option.

In [ ]:
def contains_exclusion(row, exclusion_terms):
    """Return True if any exclusion term appears in the title or abstract."""
    text = " ".join([
        str(row.get("title", "") or ""),
        str(row.get("abstract", "") or ""),
    ]).lower()
    return any(t.replace("*", "").lower() in text for t in exclusion_terms)


df_filtered = df_raw[~df_raw.apply(contains_exclusion, exclusion_terms=EXCLUSION_TERMS, axis=1)].copy()
print(f"Records after exclusion filter: {len(df_filtered)}  (removed {len(df_raw) - len(df_filtered)})")

## 6. Clean Results

> Same deduplication logic as the PubMed pipeline. I intentionally
> do **not** require a DOI here. Scholar results often lack one, and enforcing it would drop
> too many valid records. I do require a title so I have something to deduplicate on.

In [ ]:
# Drop records with no title
df_filtered = df_filtered.dropna(subset=["title"])

# Deduplicate on title (case-insensitive)
df_filtered["_title_lower"] = df_filtered["title"].str.lower().str.strip()
df_filtered = df_filtered.drop_duplicates(subset=["_title_lower"]).drop(columns=["_title_lower"])

print(f"Records after cleaning: {len(df_filtered)}")
display(df_filtered[preview_cols].head())

## 7. Export

> I tag the export with the run timestamp so I can track which run
> produced which records — important given that Scholar results are not reproducible between
> sessions.

In [ ]:
output_path = Path("google_scholar_results_2026.csv")
df_filtered.to_csv(output_path, index=False)
print(f"Exported {len(df_filtered)} records to {output_path.resolve()}")
print(f"Run timestamp: {run_ts}")